This file contains model evaluation similar to the 04 file, but condensed in a nicer way for easier comparisons.

# Entry inclusion task
## Running missing tests
The teset I already have - bert 2/3 svm 1/3 Gemini 1/3
### BERT

In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 3070


In [3]:
import torch
import pandas as pd
from transformers import BertTokenizer, BertForSequenceClassification
from tqdm import tqdm

# Load model
model_path = "Models/bert_250k_title_abstract"
tokenizer = BertTokenizer.from_pretrained(model_path)
model = BertForSequenceClassification.from_pretrained(model_path)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# Load data
df = pd.read_csv("Datasets/Inclusion_Classification/Test/400k_test.csv")
df["text"] = df["title"].fillna("") + " " + df["abstract"].fillna("")
texts = df["text"].tolist()
row_ids = df["row_id"].tolist()

# Predict
batch_size = 8
pred_labels, pred_probs = [], []

for i in tqdm(range(0, len(texts), batch_size), desc="BERT on 400k mixed"):
    batch_texts = texts[i:i+batch_size]
    inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=512, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)
    pred_labels.extend(torch.argmax(probs, dim=1).tolist())
    pred_probs.extend(probs[:, 1].tolist())

# Save
df_out = pd.DataFrame({"row_id": row_ids, "pred_label": pred_labels, "prob_1": pred_probs})
df_out.to_csv("Model_Predictions/bert_250k_on_400k_mixed.csv", index=False)
print("✅ BERT predictions saved to Model_Predictions/bert_250k_on_400k_mixed.csv")


BERT on 400k mixed: 100%|██████████| 11054/11054 [18:23<00:00, 10.02it/s]


✅ BERT predictions saved to Model_Predictions/bert_250k_on_400k_mixed.csv


### SVMs

In [5]:
import joblib
import pandas as pd

# Load model and vectorizer
vectorizer, model = joblib.load("Models/svm_250k_title_abstract.joblib")

def run_svm_prediction(input_path, output_path, use_abstract=True):
    df = pd.read_csv(input_path)
    if use_abstract:
        df["text"] = df["title"].fillna("") + " " + df["abstract"].fillna("")
    else:
        df["text"] = df["title"].fillna("")
    X = vectorizer.transform(df["text"])
    probs = model.predict_proba(X)[:, 1]
    preds = (probs >= 0.5).astype(int)
    df_out = pd.DataFrame({"row_id": df["row_id"], "svm_pred_label": preds, "svm_pred_prob": probs})
    df_out.to_csv(output_path, index=False)
    print(f"✅ SVM predictions saved to {output_path}")

# ──────────────── Run Predictions ────────────────

# 1. For 400k mixed (title + abstract)
run_svm_prediction(
    input_path="Datasets/Inclusion_Classification/Test/400k_test.csv",
    output_path="Model_Predictions/svm_250k_on_400k_mixed_preds.csv"
)

# 2. For 400k title-only
run_svm_prediction(
    input_path="Datasets/Inclusion_Classification/Test/400k_test.csv",
    output_path="Model_Predictions/svm_250k_on_400k_titles_only_preds.csv",
    use_abstract=False
)


✅ SVM predictions saved to Model_Predictions/svm_250k_on_400k_mixed_preds.csv
✅ SVM predictions saved to Model_Predictions/svm_250k_on_400k_titles_only_preds.csv


### Gemini

In [6]:
import pandas as pd
from google import genai
from tqdm import tqdm
import os

# ─── Gemini Client ─────────────────────────────
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
model_name = "gemini-2.0-flash"

# ─── Prompt Builder ─────────────────────────────
def build_prompt(text):
    return (
        "You are an expert in tropical diseases. "
        "Given the following medical research paper's title and abstract, answer only with 'Yes' or 'No':\n\n"
        f"{text}\n\n"
        "Is this paper about neglected tropical diseases?"
    )

# ─── Cost Estimation Function ──────────────────
def estimate_cost(prompt_tokens, response_tokens):
    return round((prompt_tokens * 0.00025 + response_tokens * 0.00050) / 1000, 6)

# ─── Prediction Function ───────────────────────
def run_gemini(input_path, output_path, use_abstract=True):
    df = pd.read_csv(input_path)
    df["text"] = df["title"].fillna("") + (" " + df["abstract"].fillna("") if use_abstract else "")
    results = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Gemini on {output_path}"):
        try:
            prompt = build_prompt(row["text"])
            response = client.models.generate_content(model=model_name, contents=prompt)

            answer = response.text.strip()
            label = 1 if "yes" in answer.lower() else 0

            usage = response.usage_metadata
            prompt_tokens = usage.prompt_token_count or 0
            response_tokens = usage.candidates_token_count or 0
            total_tokens = usage.total_token_count or 0
            cost = estimate_cost(prompt_tokens, response_tokens)

        except Exception as e:
            answer = str(e)
            label = -1
            prompt_tokens = response_tokens = total_tokens = 0
            cost = 0.0

        results.append({
            "row_id": row["row_id"],
            "gemini_pred": label,
            "gemini_response": answer,
            "prompt_tokens": prompt_tokens,
            "response_tokens": response_tokens,
            "total_tokens": total_tokens,
            "estimated_cost_usd": cost
        })

    df_out = pd.DataFrame(results)
    df_out.to_csv(output_path, index=False)
    print(f"✅ Gemini predictions saved to {output_path}")
    print(f"💰 Total estimated cost: ${df_out['estimated_cost_usd'].sum():.4f}")

# ─── Run Predictions ───────────────────────────

# 1. With abstracts
run_gemini(
    input_path="Datasets/Inclusion_Classification/Test/400k_test.csv",
    output_path="Model_Predictions/gemini_2-0_flash_400k_mixed_preds.csv"
)

# 2. Title-only
run_gemini(
    input_path="Datasets/Inclusion_Classification/Test/400k_test.csv",
    output_path="Model_Predictions/gemini_2-0_flash_400k_titles_only_preds.csv",
    use_abstract=False
)


Gemini on Model_Predictions/gemini_2-0_flash_400k_mixed_preds.csv:   0%|          | 179/88425 [01:04<8:47:07,  2.79it/s] 


KeyboardInterrupt: 

## Comparing results

In [9]:
import pandas as pd
from sklearn.metrics import classification_report

# ─── File Paths ─────────────────────────────────────────────
experiments = {
    "250k abstracts": {
        "SVM":    "Model_Predictions/svm_250k_title_abstract_preds.csv",
        "BERT":   "Model_Predictions/bert_250k_title_abstract_preds.csv",
        "Gemini": "Model_Predictions/gemini_2-0_flash_250k_title_abstract_preds.csv",
        "test":   "Datasets/Inclusion_Classification/Test/250k_test.csv"
    },
    "400k mixed": {
        "SVM":    "Model_Predictions/svm_250k_on_400k_mixed_preds.csv",
        "BERT":   "Model_Predictions/bert_250k_on_400k_mixed.csv",
        "Gemini": "Model_Predictions/gemini_2-0_flash_400k_mixed_preds.csv",
        "test":   "Datasets/Inclusion_Classification/Test/400k_test.csv"
    },
    "400k only titles": {
        "SVM":    "Model_Predictions/svm_250k_on_400k_titles_only_preds.csv",
        "BERT":   "Model_Predictions/bert_250k_on_400k_titles.csv",
        "Gemini": "Model_Predictions/gemini_2-0_flash_400k_titles_only_preds.csv",
        "test":   "Datasets/Inclusion_Classification/Test/400k_test.csv"
    }
}

# ─── Metric Extraction ──────────────────────────────────────
def evaluate_predictions(df_preds, df_test, pred_col, label_col='target'):
    df = df_preds.merge(df_test, on="row_id", how="left")
    y_true = df[label_col]
    y_pred = df[pred_col]
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    return {
        "Accuracy": round(report["accuracy"], 3),
        "Precision": round(report["1"]["precision"], 3),
        "Recall": round(report["1"]["recall"], 3),
        "F1 Score": round(report["1"]["f1-score"], 3)
    }

# ─── Run Evaluation ─────────────────────────────────────────
results = {}

for test_set, paths in experiments.items():
    test_df = pd.read_csv(paths["test"])
    results[test_set] = {}
    
    for model_name in ["SVM", "BERT", "Gemini"]:
        pred_path = paths.get(model_name)
        
        try:
            pred_df = pd.read_csv(pred_path)
            if model_name == "Gemini":
                pred_col = "gemini_pred"
                pred_df = pred_df[pred_df[pred_col] != -1]  # Remove failed
            else:
                pred_col = "pred_label" if "bert" in pred_path.lower() else "svm_pred_label"
            metrics = evaluate_predictions(pred_df, test_df, pred_col)
        except FileNotFoundError:
            metrics = {
                "Accuracy": "N/A",
                "Precision": "N/A",
                "Recall": "N/A",
                "F1 Score": "N/A"
            }
        results[test_set][model_name] = metrics

# ─── Format Output ──────────────────────────────────────────
for test_set, model_metrics in results.items():
    print(f"\n📊 Results for: {test_set}")
    display_df = pd.DataFrame.from_dict(model_metrics, orient="index")
    print(display_df)



📊 Results for: 250k abstracts
        Accuracy  Precision  Recall  F1 Score
SVM        0.993      0.971   0.953     0.962
BERT       0.999      0.992   0.993     0.993
Gemini     0.976      0.918   0.820     0.866

📊 Results for: 400k mixed
       Accuracy Precision Recall F1 Score
SVM       0.989     0.977  0.907    0.941
BERT       0.99     0.985  0.913    0.948
Gemini      N/A       N/A    N/A      N/A

📊 Results for: 400k only titles
       Accuracy Precision Recall F1 Score
SVM       0.976     0.981   0.77    0.863
BERT      0.973      0.98   0.74    0.843
Gemini      N/A       N/A    N/A      N/A
